In [2]:
import torch.nn as nn

In [3]:
class RNNMajorityClassifier(nn.Module):
    def __init__(self, hidden_size=8):
        super().__init__()
        self.rnn = nn.RNN(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h_n = self.rnn(x)                    # h_n: final hidden state
        return self.head(h_n[-1]).squeeze(-1)

class LSTMMajorityClassifier(nn.Module):
    def __init__(self, hidden_size=8):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, c_n) = self.lstm(x)            # LSTM also returns a cell state
        return self.head(h_n[-1]).squeeze(-1)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Define the models (from your snippet)
class RNNMajorityClassifier(nn.Module):
    def __init__(self, hidden_size=8):
        super().__init__()
        self.rnn = nn.RNN(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h_n = self.rnn(x)                    
        return self.head(h_n[-1]).squeeze(-1)

class LSTMMajorityClassifier(nn.Module):
    def __init__(self, hidden_size=8):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, c_n) = self.lstm(x)            
        return self.head(h_n[-1]).squeeze(-1)

# 2. Generate Synthetic "Majority" Data
torch.manual_seed(42) # For reproducibility

num_samples = 2000
seq_length = 5

# Create sequences of random 0s and 1s. 
# Shape needed by RNN/LSTM: (batch_size, sequence_length, input_size) -> (2000, 5, 1)
X = torch.randint(0, 2, (num_samples, seq_length, 1)).float()

# Label is 1 if the sum of the sequence is greater than half the length, else 0
y = (X.sum(dim=1).squeeze(-1) > (seq_length / 2)).float()

# Split into training and testing sets
X_train, X_test = X[:1500], X[1500:]
y_train, y_test = y[:1500], y[1500:]

# 3. Setup Training Configuration
model = LSTMMajorityClassifier(hidden_size=16)

# Since the model returns raw, unnormalized scores (no sigmoid at the end), 
# we use BCEWithLogitsLoss, which is numerically more stable than BCE + Sigmoid.
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 4. Training Loop
epochs = 200
print("Starting Training...")

for epoch in range(epochs):
    model.train() # Set to training mode
    
    # Forward pass
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    
    # Backward pass and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 40 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# 5. Evaluation
model.eval() # Set to evaluation mode
with torch.no_grad():
    # Get raw scores
    test_outputs = model(X_test)
    # Convert raw scores to probabilities using Sigmoid
    probs = torch.sigmoid(test_outputs)
    # Convert probabilities to binary predictions
    predictions = (probs > 0.5).float()
    
    accuracy = (predictions == y_test).float().mean()
    print(f"\nTest Accuracy: {accuracy.item() * 100:.2f}%")

# 6. Make a Single Prediction on New Data
print("\nMaking predictions on new sequences...")

# Two new sequences. 
# Sequence 1: Mostly 1s -> Expected Output: 1
# Sequence 2: Mostly 0s -> Expected Output: 0
new_data = torch.tensor([
    [[1.0], [0.0], [1.0], [1.0], [0.0]], 
    [[0.0], [0.0], [1.0], [0.0], [0.0]]
])

model.eval()
with torch.no_grad():
    raw_scores = model(new_data)
    probabilities = torch.sigmoid(raw_scores)
    final_preds = (probabilities > 0.5).float()

    for i in range(len(new_data)):
        seq_flat = new_data[i].view(-1).tolist()
        print(f"Sequence: {seq_flat} | Predicted Class: {int(final_preds[i].item())} (Probability: {probabilities[i].item():.4f})")

Starting Training...
Epoch [40/200], Loss: 0.1952
Epoch [80/200], Loss: 0.0308
Epoch [120/200], Loss: 0.0092
Epoch [160/200], Loss: 0.0050
Epoch [200/200], Loss: 0.0032

Test Accuracy: 100.00%

Making predictions on new sequences...
Sequence: [1.0, 0.0, 1.0, 1.0, 0.0] | Predicted Class: 1 (Probability: 0.9987)
Sequence: [0.0, 0.0, 1.0, 0.0, 0.0] | Predicted Class: 0 (Probability: 0.0000)


In [5]:
# c_n: The Cell State (Internal "Long-Term Memory")
# Shape: (num_layers=1, batch_size=2, hidden_size=4)
c_n = torch.tensor([
    [
        # Sequence 1 memory
        [ 2.841, -0.412,  5.190, -1.220],
        # Sequence 2 memory
        [-0.055,  3.774, -2.103,  0.891]
    ]
])

# h_n: The Hidden State (Gated "Working Memory" / Output)
# Shape: (num_layers=1, batch_size=2, hidden_size=4)
h_n = torch.tensor([
    [
        # Sequence 1 output
        [ 0.742, -0.183,  0.925, -0.610],
        # Sequence 2 output
        [-0.031,  0.891, -0.764,  0.422]
    ]
])

In [8]:
X_train

tensor([[[0.],
         [1.],
         [0.],
         [0.],
         [0.]],

        [[1.],
         [0.],
         [0.],
         [0.],
         [1.]],

        [[0.],
         [0.],
         [0.],
         [0.],
         [1.]],

        ...,

        [[0.],
         [1.],
         [0.],
         [0.],
         [0.]],

        [[1.],
         [0.],
         [0.],
         [1.],
         [1.]],

        [[0.],
         [1.],
         [0.],
         [0.],
         [1.]]])